# Evaluation Demo

This notebook demonstrates how to evaluate the trained Neural Receiver model.

Topics covered:
1. Loading trained model
2. Evaluating on test set
3. Analyzing detection performance (ROC curves)
4. Analyzing classification performance (confusion matrix)
5. Analyzing regression performance (parameter estimation)
6. Performance vs SNR analysis

In [ ]:
import sys
sys.path.append('..')

import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

from src.data import SignalDataset, ModulationType
from src.models import NeuralReceiver
from src.evaluation import Evaluator, MetricsCalculator
from src.evaluation.metrics import (
    plot_detection_metrics, plot_confusion_matrix, plot_regression_errors
)
from src.utils import plot_snr_performance

## 1. Load Trained Model

In [ ]:
# Path to checkpoint (update this to your trained model)
checkpoint_path = '../experiments/demo_training/checkpoints/best_model.pt'

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)

# Create model
model = NeuralReceiver(
    input_channels=2,
    base_channels=64,
    num_blocks=4,
    num_classes=12,
    dropout=0.2
)

# Load weights
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)

print(f"Loaded model from epoch {checkpoint['epoch']}")
print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")

## 2. Create Test Dataset

In [ ]:
# Create test dataset
test_dataset = SignalDataset(
    n_samples=2000,
    sequence_length=1024,
    snr_range=(-10, 0),
    no_signal_prob=0.2,
    seed=9999,  # Different seed for test
    pregenerate=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    collate_fn=SignalDataset.collate_fn
)

print(f"Test dataset: {len(test_dataset)} samples")

## 3. Evaluate Model

In [ ]:
# Create evaluator
class_names = ModulationType.get_class_names()
evaluator = Evaluator(model, device=device, class_names=class_names)

# Run evaluation
print("Evaluating model on test set...")
metrics = evaluator.evaluate(test_loader)

# Print summary
evaluator.metrics_calculator.print_summary(metrics)

## 4. Detection Performance Analysis

In [ ]:
# Plot ROC and Precision-Recall curves
plot_detection_metrics(metrics)

## 5. Classification Performance Analysis

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(metrics, class_names)

In [ ]:
# Per-class metrics
report = metrics['classification']['classification_report']

print("Per-Class Performance:")
print("="*70)
print(f"{'Class':<25} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("="*70)

for class_name in class_names:
    if class_name in report:
        r = report[class_name]
        print(f"{class_name:<25} {r['precision']:>10.3f} {r['recall']:>10.3f} {r['f1-score']:>10.3f} {r['support']:>10.0f}")

## 6. Regression Performance Analysis

In [ ]:
# Plot regression errors
plot_regression_errors(metrics)

In [ ]:
# Detailed parameter estimation errors
param_errors = metrics['regression']['param_errors']

print("Parameter Estimation Errors:")
print("="*70)
print(f"{'Parameter':<20} {'MAE':>12} {'RMSE':>12} {'Mean Error':>12} {'Std Error':>12}")
print("="*70)

for param_name, errors in param_errors.items():
    print(f"{param_name:<20} {errors['mae']:>12.4f} {errors['rmse']:>12.4f} "
          f"{errors['mean_error']:>12.4f} {errors['std_error']:>12.4f}")

## 7. Performance vs SNR Analysis

In [ ]:
# Analyze performance across different SNR levels
plot_snr_performance(
    model=model,
    dataset=SignalDataset,
    snr_range=(-15, 10),
    n_snr_points=10,
    n_samples_per_snr=100,
    device=device
)

## 8. Example Predictions

In [ ]:
# Get random test samples and show predictions
model.eval()

n_examples = 5
indices = np.random.choice(len(test_dataset), n_examples, replace=False)

print("Example Predictions:")
print("="*100)

for idx in indices:
    iq_data, labels = test_dataset[idx]
    iq_batch = iq_data.unsqueeze(0).to(device)
    
    with torch.no_grad():
        predictions = model.predict(iq_batch)
    
    true_class = labels['modulation_class'].item()
    pred_class = predictions['modulation_class'][0].item()
    
    print(f"\nSample {idx}:")
    print(f"  True: {class_names[true_class]:<25} | Pred: {class_names[pred_class]:<25} "
          f"{'✓' if true_class == pred_class else '✗'}")
    print(f"  Signal present (true): {labels['signal_present'].item():.0f}  | "
          f"Detected: {predictions['signal_detected'][0].item():.0f}  "
          f"(prob: {predictions['detection_prob'][0].item():.3f})")
    print(f"  SNR (true): {labels['snr_db'].item():>6.2f} dB | Estimated: {predictions['snr_db'][0].item():>6.2f} dB")
    
print("="*100)

## 9. Error Analysis

In [ ]:
# Analyze where the model makes mistakes
conf_matrix = metrics['classification']['confusion_matrix']

# Find most confused pairs
confused_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and conf_matrix[i, j] > 0:
            confused_pairs.append((class_names[i], class_names[j], conf_matrix[i, j]))

# Sort by confusion count
confused_pairs.sort(key=lambda x: x[2], reverse=True)

print("Most Confused Class Pairs:")
print("="*70)
print(f"{'True Class':<25} {'Predicted As':<25} {'Count':>10}")
print("="*70)

for true_class, pred_class, count in confused_pairs[:10]:
    print(f"{true_class:<25} {pred_class:<25} {count:>10.0f}")

## Summary

In this notebook, we:
1. Loaded a trained neural receiver model
2. Evaluated comprehensive metrics on test data
3. Analyzed detection performance (ROC-AUC)
4. Analyzed classification performance (accuracy, confusion matrix)
5. Analyzed regression performance (parameter estimation errors)
6. Studied performance vs SNR
7. Examined example predictions and error patterns

Key Findings:
- Detection performance (AUC-ROC): See above
- Classification accuracy: See above
- Parameter estimation quality: See regression metrics
- Performance degrades at very low SNR (< -10 dB) but remains usable

Recommendations:
- Train longer for better performance
- Consider data augmentation
- Tune hyperparameters based on error analysis
- Focus on confused class pairs for targeted improvement